# 03 - RAG Pipeline: Knowledge Base + Hybrid Retrieval + Generator + Verifier

Inputs from earlier notebooks:
- `vit_b16_partialft_best.pth` — fine-tuned ViT classifier (notebook 01)
- `classifier_bundle.pkl`     — class names + per-class thresholds

Backends supported (free-tier friendly):
- **Groq** — Llama-3.1-70B at ~700 tok/s (free key at https://console.groq.com)
- **Gemini** — `gemini-2.5-flash` (free key at https://aistudio.google.com/apikey)
- **dryrun** — prints the prompt only (for debugging the retriever)

**Two-stage workflow:** Findings first → Impression conditioned on Findings. Mirrors how radiologists actually write reports.

In [ ]:
!pip install -q sentence-transformers 'faiss-cpu>=1.9.0' rank-bm25 google-generativeai groq timm
!pip install -q transformers accelerate bitsandbytes
!pip install -q torchxrayvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, json, pickle, time, math
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np

MODEL_DIR = '/content/drive/MyDrive/Data/Model'
OUT_DIR   = '/content/drive/MyDrive/Data/iu_processed'
RAG_OUT   = '/content/drive/MyDrive/Data/rag_outputs'
os.makedirs(RAG_OUT, exist_ok=True)
print('Imports OK')

Imports OK


## 1. Knowledge Base — radiology guidelines

Four snippet types per pathology: **definition**, **feature**, **phrasing**, **pitfall**. ~50 snippets total. Sources: Radiopaedia, ACR/Fleischner Society, StatPearls. Edit/extend freely — the index rebuilds from this list.

In [ ]:
KNOWLEDGE_BASE = [
    # Atelectasis
    {'id':'atel_def','pathology':'Atelectasis','type':'definition','source':'Radiopaedia','text':'Atelectasis is collapse or incomplete expansion of lung tissue, leading to reduced gas exchange. Subtypes: obstructive, compressive, passive, adhesive, cicatricial.'},
    {'id':'atel_feat','pathology':'Atelectasis','type':'feature','source':'Radiopaedia','text':'On chest radiograph, atelectasis appears as increased opacity with associated VOLUME LOSS. Indirect signs: fissural displacement, ipsilateral mediastinal shift, hemidiaphragm elevation, crowding of bronchovascular markings.'},
    {'id':'atel_phr','pathology':'Atelectasis','type':'phrasing','source':'ACR style guide','text':'Example phrasing: "Linear opacity in the right lower lobe consistent with subsegmental atelectasis." or "Volume loss in the left lower lobe with associated opacity."'},
    {'id':'atel_pit','pathology':'Atelectasis','type':'pitfall','source':'Radiopaedia','text':'Pitfall: rounded atelectasis can mimic a mass; look for the comet-tail sign of vessels curving into the opacity and adjacent pleural thickening.'},

    # Cardiomegaly
    {'id':'card_def','pathology':'Cardiomegaly','type':'definition','source':'Radiopaedia','text':'Cardiomegaly is enlargement of the cardiac silhouette, conventionally diagnosed on PA chest radiograph when the cardiothoracic ratio (CTR) exceeds 0.5.'},
    {'id':'card_feat','pathology':'Cardiomegaly','type':'feature','source':'Radiopaedia','text':'Features include increased cardiothoracic ratio (>0.5 on PA view), widened cardiac silhouette, and possible pulmonary vascular redistribution if cardiac failure is present.'},
    {'id':'card_phr','pathology':'Cardiomegaly','type':'phrasing','source':'ACR','text':'Example phrasing: "The cardiac silhouette is enlarged with a cardiothoracic ratio of approximately 0.55, consistent with cardiomegaly."'},
    {'id':'card_pit','pathology':'Cardiomegaly','type':'pitfall','source':'Radiopaedia','text':'Pitfall: AP supine projections magnify the heart and overestimate CTR; do not call cardiomegaly on portable AP films alone.'},

    # Consolidation
    {'id':'cons_def','pathology':'Consolidation','type':'definition','source':'Radiopaedia','text':'Consolidation is replacement of air in alveoli by fluid, pus, blood, or cells, producing a region of homogeneous opacification of lung parenchyma.'},
    {'id':'cons_feat','pathology':'Consolidation','type':'feature','source':'Radiopaedia','text':'Imaging features: dense opacity that obscures vessels, air bronchograms, and a non-segmental or lobar distribution. Silhouette sign helps localise.'},
    {'id':'cons_phr','pathology':'Consolidation','type':'phrasing','source':'ACR','text':'Example phrasing: "Air-space consolidation in the right middle lobe with air bronchograms."'},
    {'id':'cons_pit','pathology':'Consolidation','type':'pitfall','source':'StatPearls','text':'Pitfall: consolidation overlaps radiologically with atelectasis and pulmonary edema; clinical context (fever, sputum) supports infectious aetiology.'},

    # Edema
    {'id':'edema_def','pathology':'Edema','type':'definition','source':'Radiopaedia','text':'Pulmonary edema is accumulation of extravascular fluid within the lungs. Most often cardiogenic, but also non-cardiogenic (ARDS, neurogenic).'},
    {'id':'edema_feat','pathology':'Edema','type':'feature','source':'Radiopaedia','text':'Findings: cephalisation of pulmonary vessels, peribronchial cuffing, Kerley B lines, perihilar "bat-wing" haze, pleural effusions, and cardiomegaly when cardiogenic.'},
    {'id':'edema_phr','pathology':'Edema','type':'phrasing','source':'ACR','text':'Example phrasing: "Findings consistent with mild pulmonary edema, including vascular cephalisation and bilateral interstitial opacities."'},
    {'id':'edema_pit','pathology':'Edema','type':'pitfall','source':'Radiopaedia','text':'Pitfall: unilateral pulmonary edema can occur with mitral regurgitation or after rapid lung re-expansion and may be mistaken for pneumonia.'},

    # Effusion
    {'id':'eff_def','pathology':'Effusion','type':'definition','source':'Radiopaedia','text':'Pleural effusion is abnormal accumulation of fluid in the pleural space. Causes include CHF, infection, malignancy, and trauma.'},
    {'id':'eff_feat','pathology':'Effusion','type':'feature','source':'Radiopaedia','text':'Features: blunting of the costophrenic angle, meniscus sign on upright views, layering on lateral decubitus. Large effusions cause mediastinal shift.'},
    {'id':'eff_phr','pathology':'Effusion','type':'phrasing','source':'ACR','text':'Example phrasing: "Blunting of the right costophrenic angle compatible with a small right pleural effusion."'},
    {'id':'eff_pit','pathology':'Effusion','type':'pitfall','source':'Radiopaedia','text':'Pitfall: subpulmonic effusion may simulate an elevated hemidiaphragm; lateral decubitus or ultrasound clarifies.'},

    # Emphysema
    {'id':'emph_def','pathology':'Emphysema','type':'definition','source':'Radiopaedia','text':'Emphysema is permanent enlargement of distal airspaces with destruction of alveolar walls. Strongly associated with chronic smoking.'},
    {'id':'emph_feat','pathology':'Emphysema','type':'feature','source':'Radiopaedia','text':'Hyperinflated lungs with flattened hemidiaphragms (>10 cm dome on PA), increased retrosternal airspace, paucity of vascular markings, and possibly bullae.'},
    {'id':'emph_phr','pathology':'Emphysema','type':'phrasing','source':'ACR','text':'Example phrasing: "Hyperinflated lungs with flattened diaphragms, suggestive of underlying emphysema."'},

    # Fibrosis
    {'id':'fib_def','pathology':'Fibrosis','type':'definition','source':'Radiopaedia','text':'Pulmonary fibrosis is scarring of lung parenchyma. UIP and NSIP are common patterns; idiopathic pulmonary fibrosis (IPF) is the prototypical UIP cause.'},
    {'id':'fib_feat','pathology':'Fibrosis','type':'feature','source':'Radiopaedia','text':'Imaging features: reticular opacities (often basal, peripheral, subpleural), traction bronchiectasis, honeycombing, and decreased lung volumes.'},
    {'id':'fib_phr','pathology':'Fibrosis','type':'phrasing','source':'ACR','text':'Example phrasing: "Coarse reticular opacities at the lung bases with associated volume loss, suggestive of pulmonary fibrosis."'},

    # Hernia (diaphragmatic / hiatal)
    {'id':'hern_def','pathology':'Hernia','type':'definition','source':'Radiopaedia','text':'Diaphragmatic hernia is herniation of abdominal contents into the thoracic cavity through a diaphragmatic defect. Hiatal hernia is herniation of stomach through the oesophageal hiatus.'},
    {'id':'hern_feat','pathology':'Hernia','type':'feature','source':'Radiopaedia','text':'Imaging clues: retrocardiac air-fluid level (hiatal hernia), bowel loops in thorax, irregular hemidiaphragm contour.'},

    # Infiltration (broad/legacy term in NIH dataset, often = consolidation/interstitial)
    {'id':'inf_def','pathology':'Infiltration','type':'definition','source':'Radiopaedia','text':'Infiltration is a broad legacy term referring to abnormal radiographic substance (fluid, cells, or fibrosis) within lung parenchyma. Modern reports prefer specific terms: consolidation, ground-glass, interstitial opacity.'},
    {'id':'inf_feat','pathology':'Infiltration','type':'feature','source':'Radiopaedia','text':'Appears as ill-defined opacity, may be focal or diffuse, with or without volume loss, and may overlap with consolidation, atelectasis, or interstitial disease.'},
    {'id':'inf_phr','pathology':'Infiltration','type':'phrasing','source':'ACR','text':'Example phrasing: "Patchy infiltrate in the right lower lobe; consider pneumonia in the appropriate clinical setting."'},

    # Mass
    {'id':'mass_def','pathology':'Mass','type':'definition','source':'Fleischner','text':'A pulmonary mass is a focal opacity larger than 3 cm. Lesions ≤3 cm are termed nodules. Always concerning for malignancy until proven otherwise.'},
    {'id':'mass_feat','pathology':'Mass','type':'feature','source':'Fleischner','text':'Imaging clues to malignancy: spiculated margin, lobulated contour, eccentric calcification, growth on serial imaging, and accompanying mediastinal lymphadenopathy.'},
    {'id':'mass_phr','pathology':'Mass','type':'phrasing','source':'ACR','text':'Example phrasing: "A 4 cm spiculated mass in the right upper lobe; CT recommended to evaluate further."'},

    # Nodule
    {'id':'nod_def','pathology':'Nodule','type':'definition','source':'Fleischner','text':'A pulmonary nodule is a rounded opacity ≤3 cm. Solid, part-solid, and ground-glass subtypes have different malignancy risks per Fleischner Society guidance.'},
    {'id':'nod_feat','pathology':'Nodule','type':'feature','source':'Fleischner','text':'Benign features: dense central or popcorn calcification, fat density (hamartoma), stability over 2 years. Suspicious features: spiculation, growth, irregular margin.'},
    {'id':'nod_phr','pathology':'Nodule','type':'phrasing','source':'ACR','text':'Example phrasing: "6 mm pulmonary nodule in the left upper lobe; per Fleischner guidelines follow-up CT in 6-12 months recommended in an average-risk patient."'},

    # Pleural Thickening
    {'id':'plt_def','pathology':'Pleural_Thickening','type':'definition','source':'Radiopaedia','text':'Pleural thickening is fibrous deposition along the pleural surface. Causes include prior infection, asbestos exposure, haemothorax, or empyema.'},
    {'id':'plt_feat','pathology':'Pleural_Thickening','type':'feature','source':'Radiopaedia','text':'Imaging: smooth or irregular pleural-based opacity that obscures the costophrenic sulcus; calcified plaques suggest asbestos-related disease.'},

    # Pneumonia
    {'id':'pna_def','pathology':'Pneumonia','type':'definition','source':'StatPearls','text':'Pneumonia is infection of the lung parenchyma. Imaging is supportive but not diagnostic; clinical correlation (fever, productive cough, leukocytosis) is required.'},
    {'id':'pna_feat','pathology':'Pneumonia','type':'feature','source':'Radiopaedia','text':'Features: lobar or segmental consolidation, air bronchograms, and possibly parapneumonic effusion. Atypical patterns include patchy or interstitial opacities.'},
    {'id':'pna_phr','pathology':'Pneumonia','type':'phrasing','source':'ACR','text':'Example phrasing: "Right lower lobe consolidation, compatible with pneumonia in the appropriate clinical setting."'},
    {'id':'pna_pit','pathology':'Pneumonia','type':'pitfall','source':'StatPearls','text':'Pitfall: imaging cannot reliably distinguish pneumonia from atelectasis or oedema; correlate with clinical findings before committing to the diagnosis.'},

    # Pneumothorax
    {'id':'ptx_def','pathology':'Pneumothorax','type':'definition','source':'Radiopaedia','text':'Pneumothorax is air in the pleural space, causing the lung to collapse partially or completely. Tension pneumothorax is a life-threatening emergency.'},
    {'id':'ptx_feat','pathology':'Pneumothorax','type':'feature','source':'Radiopaedia','text':'Imaging: a thin visceral pleural line with no lung markings beyond it; deep sulcus sign on supine films. Mediastinal shift away from the side suggests tension.'},
    {'id':'ptx_phr','pathology':'Pneumothorax','type':'phrasing','source':'ACR','text':'Example phrasing: "Visceral pleural line at the right apex with absence of lung markings laterally, consistent with a small right pneumothorax."'},
    {'id':'ptx_pit','pathology':'Pneumothorax','type':'pitfall','source':'Radiopaedia','text':'Pitfall: skin folds and clothing can mimic a pneumothorax line; the mimic line usually extends beyond the rib cage and lung markings cross it.'},

    # Generic normal-study snippet
    {'id':'normal','pathology':'Normal','type':'phrasing','source':'ACR','text':'Example phrasing for a normal study: "The lungs are clear without focal consolidation, effusion, or pneumothorax. The cardiomediastinal silhouette is normal. No acute osseous abnormality."'},
]
print('KB size:', len(KNOWLEDGE_BASE))

KB size: 48


## 2. Build embeddings + BM25 index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

EMBED_MODEL = 'pritamdeka/S-PubMedBert-MS-MARCO'  # medical-domain encoder
embedder = SentenceTransformer(EMBED_MODEL)
print('Embed dim:', embedder.get_sentence_embedding_dimension())

texts = [s['text'] for s in KNOWLEDGE_BASE]
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=16).astype('float32')
faiss_index = faiss.IndexFlatIP(emb.shape[1])
faiss_index.add(emb)

def _tok(t):
    return re.findall(r'[a-z0-9]+', t.lower())
tok_corpus = [_tok(s['text']) for s in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tok_corpus)
print('FAISS + BM25 ready')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embed dim: 768


/tmp/ipykernel_3204/4049453996.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Embed dim:', embedder.get_sentence_embedding_dimension())


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FAISS + BM25 ready


## 3. Hybrid retriever (Dense + BM25 + label-conditioned, fused with RRF)

In [ ]:
class Retriever:
    def __init__(self, label_threshold=0.3, top_n_total=8, per_label_k=5, rrf_k=60,
                 use_dense=True, use_bm25=True, use_label=True):
        self.label_threshold = label_threshold
        self.top_n_total = top_n_total
        self.per_label_k = per_label_k
        self.rrf_k = rrf_k
        self.use_dense = use_dense; self.use_bm25 = use_bm25; self.use_label = use_label

    def _dense(self, query, k):
        e = embedder.encode([query], normalize_embeddings=True).astype('float32')
        s, ix = faiss_index.search(e, k)
        return list(zip(ix[0].tolist(), s[0].tolist()))

    def _bm25(self, query, k):
        sc = bm25.get_scores(_tok(query))
        top = np.argsort(sc)[::-1][:k]
        return [(int(i), float(sc[i])) for i in top]

    def _label_match(self, label):
        norm = label.lower().replace('_', ' ')
        return [(i, 1.0) for i, s in enumerate(KNOWLEDGE_BASE)
                if s['pathology'].lower().replace('_', ' ') == norm]

    def _rrf(self, ranked_lists):
        """Reciprocal Rank Fusion."""
        scores = {}
        for ranked in ranked_lists:
            for rank, (idx, _) in enumerate(ranked):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        return sorted(scores.items(), key=lambda x: -x[1])

    def retrieve(self, predicted_labels: Dict[str, float]) -> List[dict]:
      active = [(l, p) for l, p in predicted_labels.items() if p >= self.label_threshold]
      active = sorted(active, key=lambda x: -x[1])
      if not active:
          return [s for s in KNOWLEDGE_BASE if s['pathology'] == 'Normal']

      # STRICT: only snippets whose pathology tag matches an active label
      active_labels = {l.lower().replace('_', ' ') for l, _ in active}
      candidate_idxs = [
          i for i, s in enumerate(KNOWLEDGE_BASE)
          if s['pathology'].lower().replace('_', ' ') in active_labels
      ]

      # Within that whitelist, rank by dense + BM25 fused
      if not candidate_idxs:
          return []
      query = ' '.join(l for l, _ in active) + ' chest x-ray imaging features and report phrasing'
      dense = {i: s for i, s in self._dense(query, len(KNOWLEDGE_BASE))}
      bm25  = {i: s for i, s in self._bm25(query, len(KNOWLEDGE_BASE))}
      scored = []
      for i in candidate_idxs:
          rrf = 0
          if i in dense: rrf += 1.0 / (self.rrf_k + list(dense).index(i) + 1)
          if i in bm25:  rrf += 1.0 / (self.rrf_k + list(bm25).index(i)  + 1)
          # Prefer phrasing + feature snippets over pitfall/definition
          type_boost = {'phrasing': 0.15, 'feature': 0.10, 'definition': 0.05, 'pitfall': 0.02}
          rrf += type_boost.get(KNOWLEDGE_BASE[i]['type'], 0)
          scored.append((i, rrf))
      scored.sort(key=lambda x: -x[1])
      return [KNOWLEDGE_BASE[i] for i, _ in scored[:self.top_n_total]]

retriever = Retriever(top_n_total=5, per_label_k=3)
print('Retriever ready')

Retriever ready


## 4. Generators — dryrun / Groq / Gemini

Both Groq and Gemini have generous free tiers. Get keys here:
- Groq: https://console.groq.com/keys
- Gemini: https://aistudio.google.com/apikey

**Important:** put the keys in environment variables, not in the notebook. In Colab use `userdata.get(...)` from `google.colab.userdata` or the secrets sidebar.

In [ ]:
# --- Set up API keys via Colab secrets (recommended) ---
try:
    from google.colab import userdata
    GROQ_API_KEY   = userdata.get('GROQ_API_KEY')   if userdata.get('GROQ_API_KEY')   else None
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') if userdata.get('GEMINI_API_KEY') else None
except Exception:
    GROQ_API_KEY   = os.getenv('GROQ_API_KEY')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
print('Groq key set:',   bool(GROQ_API_KEY))
print('Gemini key set:', bool(GEMINI_API_KEY))

Groq key set: False
Gemini key set: False


In [ ]:
SYSTEM_PROMPT = """You are an experienced radiologist drafting a chest X-ray report.

STRICT RULES:
1. Mention ONLY abnormalities in the PREDICTED ABNORMALITIES list. Do not introduce conditions from the retrieved context that are not in the predicted list.
2. The retrieved guideline context is for STYLE and PHRASING reference only — use it to write about predicted conditions, not to add new ones.
3. For any predicted abnormality with probability < 0.5, use hedging ("possible", "cannot be excluded", "suggestive of").
4. If a condition is absent, use clear negation ("no evidence of pneumothorax", "no consolidation identified").

Generate two sections:
1. FINDINGS — systematic description (lungs, heart, mediastinum, pleura, bones, soft tissues)
2. IMPRESSION — one or two sentences synthesising the most salient findings."""

def build_user_prompt(predicted_labels: Dict[str, float], snippets: List[dict]) -> str:
    label_lines = '\n'.join(
        f'  - {lbl}: {p:.2f}'
        for lbl, p in sorted(predicted_labels.items(), key=lambda x: -x[1])
        if p >= 0.2
    ) or '  (no labels above 0.20 — likely a normal study)'
    snippet_blocks = '\n\n'.join(
        f"[{s['pathology']} | {s['type']} | source: {s['source']}]\n{s['text']}"
        for s in snippets
    )
    return (
        f'PREDICTED ABNORMALITIES (from vision model, with probabilities):\n{label_lines}\n\n'
        f'RETRIEVED RADIOLOGY GUIDELINE CONTEXT:\n{snippet_blocks}\n\n'
        'Generate the FINDINGS section, then the IMPRESSION section, of the radiology report.'
    )

In [ ]:
import time, re

class Generator:
    """
    Multi-backend generator with automatic fallback:
      Groq → Gemini → template (rule-based, never fails).
    Built-in throttling and 429 handling.
    """
    def __init__(self, backend='auto', model=None, api_key=None,
                 temperature=0.2, max_tokens=300):
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.requested_backend = backend
        self.model_name_used = '(template)'
        self.last_call_time = 0.0
        self.min_gap_sec = 4.5     # ~13 RPM ceiling
        self._init_backends(api_key)
        self._fallback_log = []

    def _init_backends(self, api_key):
        self.groq_client = None; self.gemini_client = None
        try:
            if GROQ_API_KEY:
                from groq import Groq
                self.groq_client = Groq(api_key=api_key or GROQ_API_KEY)
                self.groq_model  = 'llama-3.3-70b-versatile'
        except Exception as e:
            print('Groq init failed:', e)
        try:
            if GEMINI_API_KEY:
                import google.generativeai as genai
                genai.configure(api_key=api_key or GEMINI_API_KEY)
                self.gemini_model = 'gemini-2.5-flash-lite'
                self.gemini_client = genai.GenerativeModel(
                    self.gemini_model, system_instruction=SYSTEM_PROMPT)
        except Exception as e:
            print('Gemini init failed:', e)
        self.hf_pipe = None
        try:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
            self.hf_model_name = 'Qwen/Qwen2.5-7B-Instruct'   # ~14 GB fp16 on A100
            tok = AutoTokenizer.from_pretrained(self.hf_model_name)
            mdl = AutoModelForCausalLM.from_pretrained(
                self.hf_model_name,
                torch_dtype=torch.float16,
                device_map='auto',
            )
            self.hf_pipe = pipeline('text-generation', model=mdl, tokenizer=tok)
            print('Loaded local model:', self.hf_model_name)
        except Exception as e:
            print('HF local init failed:', e)
        self.backend = (
            'hf_local' if self.hf_pipe else
            'groq' if self.groq_client else
            'gemini' if self.gemini_client else
            'template'
        )

    # ---------- backend calls ----------
    def _throttle(self):
        gap = self.min_gap_sec - (time.time() - self.last_call_time)
        if gap > 0:
            time.sleep(gap)
        self.last_call_time = time.time()
    def _call_hf_local(self, user_prompt):
      messages = [{'role': 'system', 'content': SYSTEM_PROMPT},{'role': 'user',   'content': user_prompt},]
      out = self.hf_pipe(
          messages,
          max_new_tokens=400,
          min_new_tokens=150,           # force at least 150 tokens of report
          do_sample=True,
          temperature=0.3,
          top_p=0.9,
          repetition_penalty=1.05,
          return_full_text=False,
      )
      gen = out[0]['generated_text']
      if isinstance(gen, list):
          self.model_name_used = self.hf_model_name
          return gen[-1]['content']
      return str(gen)
      def _call_groq(self, user_prompt):
          self._throttle()
          resp = self.groq_client.chat.completions.create(
              model=self.groq_model, temperature=self.temperature,
              max_tokens=self.max_tokens,
              messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user_prompt}])
          self.model_name_used = self.groq_model
          return resp.choices[0].message.content

    def _call_gemini(self, user_prompt):
        import google.generativeai as genai
        self._throttle()
        cfg = genai.types.GenerationConfig(
            temperature=self.temperature, max_output_tokens=self.max_tokens)
        resp = self.gemini_client.generate_content(user_prompt, generation_config=cfg)
        self.model_name_used = self.gemini_model
        return resp.text

    def _template_report(self, predicted_labels, snippets):
        """Rule-based fallback: assemble a minimal report from labels + snippets.
        Not pretty, but BLEU/ROUGE-evaluable and deterministic."""
        confident = [(l,p) for l,p in predicted_labels.items() if p >= 0.4]
        confident.sort(key=lambda x: -x[1])
        possible  = [(l,p) for l,p in predicted_labels.items() if 0.2 <= p < 0.4]
        possible.sort(key=lambda x: -x[1])
        find_lines = []
        for lbl, prob in confident:
            phr = next((s['text'] for s in snippets
                        if s['pathology']==lbl and s['type']=='phrasing'), None)
            if phr is None:
                phr = f'Findings consistent with {lbl.lower().replace("_"," ")}.'
            find_lines.append(phr)
        for lbl, prob in possible:
            find_lines.append(f'Possible {lbl.lower().replace("_"," ")}; cannot be excluded.')
        if not find_lines:
            find_lines = ['The lungs are clear without focal consolidation, effusion, or pneumothorax. The cardiomediastinal silhouette is normal.']
        impression = (', '.join(l.lower().replace('_',' ') for l,_ in confident)
                      if confident else 'No acute cardiopulmonary findings.')
        self.model_name_used = 'template'
        return f"FINDINGS: {' '.join(find_lines)}\n\nIMPRESSION: {impression.capitalize()}."

    # ---------- public API ----------
    def generate(self, predicted_labels, snippets):
        user_prompt = build_user_prompt(predicted_labels, snippets)

        # Build backend priority list
        backends = []
        if self.hf_pipe:       backends.append('hf_local')
        if self.groq_client:   backends.append('groq')
        if self.gemini_client: backends.append('gemini')

        # Dispatch loop — try each backend, retry on 429
        for backend in backends:
            for attempt in range(2):
                try:
                    if backend == 'hf_local': return self._call_hf_local(user_prompt)
                    if backend == 'groq':     return self._call_groq(user_prompt)
                    if backend == 'gemini':   return self._call_gemini(user_prompt)
                except Exception as e:
                    msg = str(e)
                    is_429 = '429' in msg or 'rate_limit' in msg.lower() or 'quota' in msg.lower()
                    if is_429:
                        m = re.search(r'(?:retry in|try again in) ?(\d+(?:\.\d+)?)s', msg)
                        wait = min(float(m.group(1))+2 if m else 30, 90)
                        print(f'  [{backend} 429] sleep {wait:.0f}s')
                        time.sleep(wait)
                        continue
                    print(f'  [{backend} error] {msg[:160]}')
                    break  # try next backend

        # All backends failed — deterministic template fallback
        self._fallback_log.append('template')
        return self._template_report(predicted_labels, snippets)

# Build it — backend='auto' picks the best available
generator = Generator(backend='auto')
print('Active backend:', generator.backend)
print('Models:',
      'groq=', getattr(generator, 'groq_model', None),
      '| gemini=', getattr(generator, 'gemini_model', None))

# Pick whichever backend you have a key for
if GEMINI_API_KEY:
    generator = Generator(backend='gemini')
elif GROQ_API_KEY:
    generator = Generator(backend='groq')
else:
    generator = Generator(backend='dryrun')
print('Generator backend:', generator.backend, '|', getattr(generator, 'model', ''))

## 5. Verifier — hallucination check (regex labeller, CheXpert-lite)

Counts conditions mentioned in the generated text vs. conditions the classifier predicted. Three numbers per report:
- **hallucinated** = mentioned but not predicted (with high confidence)
- **missed**       = predicted with high confidence but not mentioned
- **is_consistent** = no hallucinations

In [ ]:
SYNONYMS = {
    'Atelectasis':         [r'atelectas', r'\bcollapse\b'],
    'Cardiomegaly':        [r'cardiomegaly', r'enlarged heart', r'cardiac enlargement', r'cardiothoracic ratio'],
    'Consolidation':       [r'consolidat'],
    'Edema':               [r'\bedema\b', r'pulmonary edema', r'kerley'],
    'Effusion':            [r'effusion', r'pleural fluid'],
    'Emphysema':           [r'emphysema', r'hyperinflat'],
    'Fibrosis':            [r'fibrosis', r'fibrotic', r'honeycomb'],
    'Hernia':              [r'hernia'],
    'Infiltration':        [r'infiltrat'],
    'Mass':                [r'\bmass\b'],
    'Nodule':              [r'nodule'],
    'Pleural_Thickening':  [r'pleural thickening'],
    'Pneumonia':           [r'pneumonia'],
    'Pneumothorax':        [r'pneumothorax'],
}

NEG_RE = re.compile(r'\b(no|without|absent|absence of|negative for|free of|denies|rule out|r/o|not appreciated|not identified|not seen|not visualised|no evidence of|no significant|no acute)\b')

def verify(report: str, predicted_labels: Dict[str, float], threshold=0.4) -> dict:
    text = report.lower()
    confident = {l for l, p in predicted_labels.items() if p >= threshold}
    mentioned = set()
    for label, pats in SYNONYMS.items():
        for pat in pats:
            for m in re.finditer(pat, text):
                # Skip mentions negated within ~50 chars to the left
                window = text[max(0, m.start()-50):m.start()]
                if NEG_RE.search(window):
                    continue
                mentioned.add(label)
                break
            if label in mentioned:
                break
    halluc = mentioned - confident
    missed = confident - mentioned
    return {'mentioned': sorted(mentioned),
            'predicted_confident': sorted(confident),
            'hallucinated': sorted(halluc),
            'missed': sorted(missed),
            'is_consistent': len(halluc) == 0}

## 6. Wire in the fine-tuned ViT classifier (from notebook 01)

In [ ]:
import torch, torch.nn as nn, timm
from torchvision import transforms
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(os.path.join(MODEL_DIR, 'classifier_bundle.pkl'), 'rb') as f:
    BUNDLE = pickle.load(f)
CLASS_NAMES = BUNDLE['class_names']
THRESHOLDS  = np.array(BUNDLE['thresholds'])
CKPT_PATH   = BUNDLE['best_ckpt_path']
print('Loaded classifier bundle. Mean test AUC:', BUNDLE['mean_auc'])

class ViTPartialFT(nn.Module):
    def __init__(self, num_classes=14, unfreeze_last_n=4):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=num_classes)
    def forward(self, x):
        return self.backbone(x)

vit = ViTPartialFT(num_classes=len(CLASS_NAMES)).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
vit.load_state_dict(state, strict=False)
vit.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_labels_for_image(image_path: str) -> Dict[str, float]:
    img = Image.open(image_path).convert('RGB')
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    logits = vit(x)
    probs = torch.sigmoid(logits)[0].cpu().numpy()
    return dict(zip(CLASS_NAMES, probs.tolist()))

Loaded classifier bundle. Mean test AUC: 0.7579517975602944


In [ ]:
# === TXV-based predictor (better for IU because TXV was pretrained on OpenI) ===
import torchxrayvision as xrv
from torchvision import transforms as T

txv_model = xrv.models.DenseNet(weights='densenet121-res224-all').to(DEVICE).eval()
TXV_LABELS = txv_model.pathologies

TXV_TO_OURS = {c: c for c in CLASS_NAMES}  # all 14 class names already match TXV

txv_preprocess = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Lambda(lambda x: (x * 2 - 1) * 1024),  # TXV expects [-1024, 1024]
])

@torch.no_grad()
def predict_labels_txv(image_path: str) -> Dict[str, float]:
    img = Image.open(image_path).convert('RGB')
    x = txv_preprocess(img).unsqueeze(0).to(DEVICE)
    out = torch.sigmoid(txv_model(x))[0].cpu().numpy()  # (18,)
    return {c: float(out[TXV_LABELS.index(TXV_TO_OURS[c])]) for c in CLASS_NAMES}

print('TXV predictor ready. Labels match:', all(c in TXV_LABELS for c in CLASS_NAMES))

In [ ]:
def predict_with_normal_override(image_path: str, predictor=predict_labels_txv,
                                  abnormal_threshold=0.4) -> Dict[str, float]:
    preds = predictor(image_path)
    top = max(preds.values())
    if top < abnormal_threshold:
        # Force "normal study" — clip everything below threshold
        return {c: 0.0 for c in CLASS_NAMES}
    return preds

## 7. End-to-end pipeline

In [ ]:
def run_pipeline(predicted_labels, retriever, generator, with_verifier_loop=False):
    snippets = retriever.retrieve(predicted_labels)
    report = generator.generate(predicted_labels, snippets)
    v = verify(report, predicted_labels)
    if with_verifier_loop and not v['is_consistent']:
        # Regenerate once with a corrective hint
        hint = (f"\n\nNote: do NOT mention these labels (model did not predict them with confidence): {v['hallucinated']}."
                f" The report should mention: {v['predicted_confident']}.")
        snippets2 = retriever.retrieve(predicted_labels)
        old_prompt = build_user_prompt(predicted_labels, snippets2)
        # Reuse generator with augmented user prompt by passing snippets again — for simplicity we just call generate again
        report2 = generator.generate(predicted_labels, snippets2)
        v2 = verify(report2, predicted_labels)
        if len(v2['hallucinated']) <= len(v['hallucinated']):
            report, v = report2, v2
    return {
        'predicted_labels': predicted_labels,
        'retrieved_snippets': snippets,
        'report': report,
        'verifier': v,
    }

## 8. Demo run

In [ ]:
# Pick one IU test image and run the full pipeline on it.
import pandas as pd
iu_test = pd.read_csv(os.path.join(OUT_DIR, 'iu_test.csv'))
row = iu_test.iloc[0]
print('Image:', row['image_path'])
preds = predict_with_normal_override(row['image_path'])
print('\nTop-5 predictions:')
for l, p in sorted(preds.items(), key=lambda x: -x[1])[:5]:
    print(f'  {l:22s} {p:.3f}')
result = run_pipeline(preds, retriever, generator)
print('\n--- Generated report ---\n')
print(result['report'])
print('\n--- Verifier ---\n', result['verifier'])
print('\n--- Ground-truth Findings (IU) ---\n', row['findings'])
print('\n--- Ground-truth Impression (IU) ---\n', row['impression'])

Image: /content/drive/MyDrive/Data/iu_images/2_IM-0652-1001.dcm.png

Top-5 predictions:
  Effusion               0.622
  Cardiomegaly           0.345
  Infiltration           0.306
  Pleural_Thickening     0.294
  Consolidation          0.238

--- Generated report ---

=== SYSTEM ===
You are an experienced radiologist drafting a chest X-ray report.

Generate the report in two sections:
1. FINDINGS: Describe imaging findings systematically (lungs, heart, mediastinum, pleura, bones, soft tissues). Reference only the predicted abnormalities and the provided guideline context. Do NOT invent findings not supported by either.
2. IMPRESSION: Provide a concise clinical conclusion based strictly on the Findings.

Use formal radiology phrasing. If a predicted abnormality has low probability, use hedging language ("possible", "cannot be excluded", "suggestive of"). If the predicted labels indicate a normal study, state that clearly.

=== USER ===
PREDICTED ABNORMALITIES (from vision model, with p